In [ ]:
'''
We have to generate lag based features and rolling statistics features to set a baseline for 
MLForecast

'''

In [1]:
import numpy as np
import pandas as pd 
import polars as pl
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from statsmodels.tsa.stattools import acf, pacf
import json


In [2]:
import plotly.express as px

In [3]:
with open('forecast_set_columns.json', 'r') as f:
    columns_dict = json.load(f)

In [4]:
dp = pl.scan_parquet('forecasting-data-set.parquet')

In [5]:
dp.schema

C:\Users\tejas\AppData\Local\Temp\ipykernel_11092\1898287363.py:1: PerformanceWarning: Resolving the schema of a LazyFrame is a potentially expensive operation. Use `LazyFrame.collect_schema()` to get the schema without this warning.
  dp.schema


Schema([('ProvinceID', Float64),
        ('NOC_GroupingID', Float64),
        ('datestamp', Datetime(time_unit='ns', time_zone=None)),
        ('Full-time employment', Float64),
        ('y', Float64),
        ('unique_id', Int64),
        ('month', Int32),
        ('quarter', Int32),
        ('month_in_quarter', Int32),
        ('month_in_quarter_sin', Float64),
        ('month_in_quarter_cos', Float64),
        ('month_sin', Float64),
        ('month_cos', Float64),
        ('quarter_sin', Float64),
        ('quarter_cos', Float64),
        ('week', Int32),
        ('week_sin', Float64),
        ('week_cos', Float64)])

In [6]:
columns_dict

{'Id': ['unique_id'],
 'Target': ['y'],
 'Datestamp': ['datestamp'],
 'ExogenousFeature': ['month_in_quarter',
  'month_in_quarter_sin',
  'month_in_quarter_cos',
  'month_sin',
  'month_cos',
  'quarter_sin',
  'quarter_cos',
  'week',
  'week_sin',
  'week_cos']}

In [7]:
ids = columns_dict['Id']
ds = columns_dict['Datestamp']
target = columns_dict['Target']
exog = columns_dict['ExogenousFeature']

all_cols = ids + ds + target + exog

all_cols

['unique_id',
 'datestamp',
 'y',
 'month_in_quarter',
 'month_in_quarter_sin',
 'month_in_quarter_cos',
 'month_sin',
 'month_cos',
 'quarter_sin',
 'quarter_cos',
 'week',
 'week_sin',
 'week_cos']

In [8]:
df_Cyclical_Encoding = dp.select(ds + exog).unique().collect().to_pandas()
df_Cyclical_Encoding

,datestamp,month_in_quarter,month_in_quarter_sin,month_in_quarter_cos,month_sin,month_cos,quarter_sin,quarter_cos,week,week_sin,week_cos
0,2001-10-01,1,8.660254e-01,-0.5,-8.660254e-01,5.000000e-01,-2.449294e-16,1.000000e+00,40,-9.927089e-01,0.120537
1,2024-09-01,3,-2.449294e-16,1.0,-1.000000e+00,-1.836970e-16,-1.000000e+00,-1.836970e-16,35,-8.854560e-01,-0.464723
2,2012-02-01,2,-8.660254e-01,-0.5,8.660254e-01,5.000000e-01,1.000000e+00,6.123234e-17,5,5.680647e-01,0.822984
3,2017-07-01,1,8.660254e-01,-0.5,-5.000000e-01,-8.660254e-01,-1.000000e+00,-1.836970e-16,26,-3.216245e-16,-1.000000
4,2005-12-01,3,-2.449294e-16,1.0,-2.449294e-16,1.000000e+00,-2.449294e-16,1.000000e+00,48,-4.647232e-01,0.885456
...,...,...,...,...,...,...,...,...,...,...,...
304,2006-01-01,1,8.660254e-01,-0.5,5.000000e-01,8.660254e-01,1.000000e+00,6.123234e-17,52,6.432491e-16,1.000000
305,2021-07-01,1,8.660254e-01,-0.5,-5.000000e-01,-8.660254e-01,-1.000000e+00,-1.836970e-16,26,-3.216245e-16,-1.000000
306,2024-02-01,2,-8.660254e-01,-0.5,8.660254e-01,5.000000e-01,1.000000e+00,6.123234e-17,5,5.680647e-01,0.822984
307,2016-01-01,1,8.660254e-01,-0.5,5.000000e-01,8.660254e-01,1.000000e+00,6.123234e-17,53,1.205367e-01,0.992709


In [9]:
df_Cyclical_Encoding = df_Cyclical_Encoding.sort_values(by=['datestamp'], ascending=True)
df_Cyclical_Encoding

,datestamp,month_in_quarter,month_in_quarter_sin,month_in_quarter_cos,month_sin,month_cos,quarter_sin,quarter_cos,week,week_sin,week_cos
119,2000-01-01,1,8.660254e-01,-0.5,5.000000e-01,8.660254e-01,1.000000e+00,6.123234e-17,52,6.432491e-16,1.000000e+00
81,2000-02-01,2,-8.660254e-01,-0.5,8.660254e-01,5.000000e-01,1.000000e+00,6.123234e-17,5,5.680647e-01,8.229839e-01
32,2000-03-01,3,-2.449294e-16,1.0,1.000000e+00,6.123234e-17,1.000000e+00,6.123234e-17,9,8.854560e-01,4.647232e-01
257,2000-04-01,1,8.660254e-01,-0.5,8.660254e-01,-5.000000e-01,1.224647e-16,-1.000000e+00,13,1.000000e+00,-1.608123e-16
77,2000-05-01,2,-8.660254e-01,-0.5,5.000000e-01,-8.660254e-01,1.224647e-16,-1.000000e+00,18,8.229839e-01,-5.680647e-01
...,...,...,...,...,...,...,...,...,...,...,...
300,2025-05-01,2,-8.660254e-01,-0.5,5.000000e-01,-8.660254e-01,1.224647e-16,-1.000000e+00,18,8.229839e-01,-5.680647e-01
174,2025-06-01,3,-2.449294e-16,1.0,1.224647e-16,-1.000000e+00,1.224647e-16,-1.000000e+00,22,4.647232e-01,-8.854560e-01
196,2025-07-01,1,8.660254e-01,-0.5,-5.000000e-01,-8.660254e-01,-1.000000e+00,-1.836970e-16,27,-1.205367e-01,-9.927089e-01
160,2025-08-01,2,-8.660254e-01,-0.5,-8.660254e-01,-5.000000e-01,-1.000000e+00,-1.836970e-16,31,-5.680647e-01,-8.229839e-01


In [ ]:
df_Cyclical_Encoding = df_Cyclical_Encoding.drop(['month_in_quarter','week'], axis=1)


In [12]:
print(df_Cyclical_Encoding)

     datestamp  month_in_quarter_sin  month_in_quarter_cos     month_sin  \
119 2000-01-01          8.660254e-01                  -0.5  5.000000e-01   
81  2000-02-01         -8.660254e-01                  -0.5  8.660254e-01   
32  2000-03-01         -2.449294e-16                   1.0  1.000000e+00   
257 2000-04-01          8.660254e-01                  -0.5  8.660254e-01   
77  2000-05-01         -8.660254e-01                  -0.5  5.000000e-01   
..         ...                   ...                   ...           ...   
300 2025-05-01         -8.660254e-01                  -0.5  5.000000e-01   
174 2025-06-01         -2.449294e-16                   1.0  1.224647e-16   
196 2025-07-01          8.660254e-01                  -0.5 -5.000000e-01   
160 2025-08-01         -8.660254e-01                  -0.5 -8.660254e-01   
216 2025-09-01         -2.449294e-16                   1.0 -1.000000e+00   

        month_cos   quarter_sin   quarter_cos      week_sin      week_cos  
119  8.6602

In [13]:

df = df_Cyclical_Encoding.sort_values("datestamp")

In [21]:
fig = make_subplots(
    rows=4, cols=1,
    shared_xaxes=True,
    subplot_titles=[
        "Month in Quarter (Sin/Cos)",
        "Month Seasonality (Sin/Cos)",
        "Quarter Seasonality (Sin/Cos)",
        "Week Seasonality (Sin/Cos)"
    ],
    vertical_spacing=0.07
)

# Month in Quarter
fig.add_trace(
    go.Scatter(x=df["datestamp"], y=df["month_in_quarter_sin"],
               mode="lines", name="month_in_quarter_sin"),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=df["datestamp"], y=df["month_in_quarter_cos"],
               mode="lines", name="month_in_quarter_cos"),
    row=1, col=1
)

# Month
fig.add_trace(
    go.Scatter(x=df["datestamp"], y=df["month_sin"],
               mode="lines", name="month_sin"),
    row=2, col=1
)
fig.add_trace(
    go.Scatter(x=df["datestamp"], y=df["month_cos"],
               mode="lines", name="month_cos"),
    row=2, col=1
)

# Quarter
fig.add_trace(
    go.Scatter(x=df["datestamp"], y=df["quarter_sin"],
               mode="lines", name="quarter_sin"),
    row=3, col=1
)
fig.add_trace(
    go.Scatter(x=df["datestamp"], y=df["quarter_cos"],
               mode="lines", name="quarter_cos"),
    row=3, col=1
)

# Week
fig.add_trace(
    go.Scatter(x=df["datestamp"], y=df["week_sin"],
               mode="lines", name="week_sin"),
    row=4, col=1
)
fig.add_trace(
    go.Scatter(x=df["datestamp"], y=df["week_cos"],
               mode="lines", name="week_cos"),
    row=4, col=1
)

fig.update_layout(
    height=900,
    width=2000,
    title="Cyclical Time Feature Encodings",
    showlegend=True,
    plot_bgcolor="#B4C2D4", 
    
)

fig.show()

In [22]:
fig.write_image('Cyclical_Encoding.svg')

In [107]:
df_Reduced = dp.select(all_cols).collect().to_pandas().sort_values(by=['unique_id','datestamp'], ascending=True)
df_Reduced

,unique_id,datestamp,y,month_in_quarter,month_in_quarter_sin,month_in_quarter_cos,month_sin,month_cos,quarter_sin,quarter_cos,week,week_sin,week_cos
4944,1,2000-01-01,12.9,1,8.660254e-01,-0.5,5.000000e-01,8.660254e-01,1.000000e+00,6.123234e-17,52,6.432491e-16,1.000000e+00
4945,1,2000-02-01,14.2,2,-8.660254e-01,-0.5,8.660254e-01,5.000000e-01,1.000000e+00,6.123234e-17,5,5.680647e-01,8.229839e-01
4946,1,2000-03-01,12.0,3,-2.449294e-16,1.0,1.000000e+00,6.123234e-17,1.000000e+00,6.123234e-17,9,8.854560e-01,4.647232e-01
4947,1,2000-04-01,12.2,1,8.660254e-01,-0.5,8.660254e-01,-5.000000e-01,1.224647e-16,-1.000000e+00,13,1.000000e+00,-1.608123e-16
4948,1,2000-05-01,11.8,2,-8.660254e-01,-0.5,5.000000e-01,-8.660254e-01,1.224647e-16,-1.000000e+00,18,8.229839e-01,-5.680647e-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...
69520,554,2025-05-01,18.9,2,-8.660254e-01,-0.5,5.000000e-01,-8.660254e-01,1.224647e-16,-1.000000e+00,18,8.229839e-01,-5.680647e-01
69521,554,2025-06-01,21.8,3,-2.449294e-16,1.0,1.224647e-16,-1.000000e+00,1.224647e-16,-1.000000e+00,22,4.647232e-01,-8.854560e-01
69522,554,2025-07-01,22.3,1,8.660254e-01,-0.5,-5.000000e-01,-8.660254e-01,-1.000000e+00,-1.836970e-16,27,-1.205367e-01,-9.927089e-01
69523,554,2025-08-01,24.5,2,-8.660254e-01,-0.5,-8.660254e-01,-5.000000e-01,-1.000000e+00,-1.836970e-16,31,-5.680647e-01,-8.229839e-01


Conducting Prelimnary ACF and PACF analysis

In [54]:
def plot_ts_acf_pacf(series, unique_id, lags=48):

    # Calculate ACF and PACF
    acf_vals = acf(series, nlags=lags)
    pacf_vals = pacf(series, nlags=lags)

    lag_range = np.arange(len(acf_vals))

    # Significance bounds
    conf = 1.96 / np.sqrt(len(series))

    fig = make_subplots(
        rows=2, cols=2,
        shared_xaxes=False,
        column_widths=[0.5, 0.5],
        row_heights=[0.6, 0.4],  # first row taller
        specs=[[{"colspan": 2}, None],
               [{}, {}]],
       subplot_titles=(f"Original Time Series", "PACF", "ACF"),
       vertical_spacing=0.07
    )

    fig.add_trace(
        go.Scatter(
        x=series.index,
        y=series,
        mode='lines',
        line=dict(color="#245087", width=2.5),
        name='Residuals'
    ),
    row=1, col=1

)
    conf_int = 1.96/np.sqrt(len(series))

    fig.add_trace(
    go.Bar(
        x=np.arange(len(acf_vals)),
        y=pacf_vals,
        name='ACF',
        marker_color="#245087",
    ),
    row=2, col=1
    )
    fig.add_hline(y=conf_int, line_dash="dot", line_color="red", row=2, col=1)
    fig.add_hline(y=-conf_int, line_dash="dot", line_color="red", row=2, col=1)

    fig.add_trace(
    go.Bar(
        x=np.arange(len(acf_vals)),
        y=acf_vals,
        name='ACF',
        marker_color="#245087",
    ),
    row=2, col=2
    )

    
    fig.add_hline(y=conf_int, line_dash="dot", line_color="red", row=2, col=2)
    fig.add_hline(y=-conf_int, line_dash="dot", line_color="red", row=2, col=2)

    fig.update_layout(
    height=900,
    width=1000,
    title_text=f"ACF and PACF Analysis for Series: {unique_id}",
    showlegend=False,
)


    

    return fig

In [55]:
unique_id = 1
Sample = df_Reduced[df_Reduced['unique_id']==unique_id]

AnalysisPlot_1 = plot_ts_acf_pacf(series=Sample['y'], unique_id=unique_id)

In [56]:
AnalysisPlot_1.show()

In [69]:
AnalysisPlot_1.write_image('ACF_PACF_Analysis_Series_1.svg')

In [ ]:
'''
From the plot above,the PACF plot indicates the first lag is highly important then tapers off, whereas in teh ACF plot, there 
is a slow decay, meaning that rolling statistics going back far would help capture the long term temporal dependcies. 

'''

In [67]:
unique_id = 70
Sample = df_Reduced[df_Reduced['unique_id']==unique_id]

AnalysisPlot_70 = plot_ts_acf_pacf(series=Sample['y'], unique_id=unique_id)

In [68]:
AnalysisPlot_70.show()

In [70]:
AnalysisPlot_70.write_image(f'ACF_PACF_Analysis_Series_{unique_id}.svg')

In [71]:
unique_id = 530
Sample = df_Reduced[df_Reduced['unique_id']==unique_id]

AnalysisPlot_530 = plot_ts_acf_pacf(series=Sample['y'], unique_id=unique_id)

In [72]:
AnalysisPlot_530.show()

In [73]:
AnalysisPlot_530.write_image(f'ACF_PACF_Analysis_Series_{unique_id}.svg')

Have to run a batch algorithm for ACF and PACF corrs and store them in dataframe

In [74]:
from tqdm import tqdm

In [75]:
max_lag = 48 
unique_ids = sorted(df_Reduced['unique_id'].unique().tolist())
acf_pacf_results = []

for uid in tqdm(unique_ids, desc='Conducting ACF and PACF', unit='Series'):
    series = df_Reduced[df_Reduced['unique_id']==uid]['y']

    acf_vals = acf(series, nlags=max_lag)
    pacf_vals = pacf(series, nlags=max_lag)

    for lag in range(len(acf_vals)):
        acf_pacf_results.append({
            "unique_id": uid,
            "lag": lag,
            "acf": acf_vals[lag],
            "pacf": pacf_vals[lag]
        })


Conducting ACF and PACF: 100%|██████████| 554/554 [00:01<00:00, 312.37Series/s]


In [76]:
df_ACF_And_Pacf_Correlations = pd.DataFrame(acf_pacf_results)

In [78]:
df_ACF_And_Pacf_Correlations = df_ACF_And_Pacf_Correlations[df_ACF_And_Pacf_Correlations['lag'] > 0]
df_ACF_And_Pacf_Correlations

,unique_id,lag,acf,pacf
1,1,1,0.941245,0.944301
2,1,2,0.906149,0.187894
3,1,3,0.868758,0.004017
4,1,4,0.839125,0.054382
5,1,5,0.810076,0.018753
...,...,...,...,...
27141,554,44,0.246769,-0.044377
27142,554,45,0.257806,0.026829
27143,554,46,0.262059,0.123376
27144,554,47,0.268875,0.074215


In [80]:
df_ACF_And_Pacf_Correlations['Lag'] = df_ACF_And_Pacf_Correlations['lag'].apply(lambda x: f"Lag_{x}")

C:\Users\tejas\AppData\Local\Temp\ipykernel_40496\640673741.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_ACF_And_Pacf_Correlations['Lag'] = df_ACF_And_Pacf_Correlations['lag'].apply(lambda x: f"Lag_{x}")


In [100]:
Lag_Importance_Correlations = px.box(
    data_frame=df_ACF_And_Pacf_Correlations,
    title='PACF Correlation Distribution Across Lags',
    x='Lag',
    y='pacf',
    width=1000,
    height=900,
    color_discrete_sequence=['#184C96']
)

conf_int = 1.96/np.sqrt(len(df_Reduced['datestamp'].unique()))

Lag_Importance_Correlations.add_hline(y=conf_int, line_dash="dot", line_color="red")
Lag_Importance_Correlations.add_hline(y=-conf_int, line_dash="dot", line_color="red")

Lag_Importance_Correlations.update_layout(
     title_font=dict(size=16),  # Set the title font size
    font=dict(size=15),  # Set general font size for labels and axis titles
            hoverlabel=dict(
                font_size=20  # Set font size for hover data
            ),
            yaxis=dict(
                title='PACF Significance Distrbution',
                title_font=dict(size=18),  # Set y-axis title font size
                tickfont=dict(size=12)  # Set y-axis tick font size
            ),
            xaxis=dict(
                title='Lags',
                title_font=dict(size=18),  # Set x-axis title font size
                tickfont=dict(size=12)  # Set x-axis tick font size
            ),
             legend=dict(
                orientation='h',       
                yanchor='bottom',     
                y=-0.3,               
                xanchor='center',      
                 x=0.5,                 
                font=dict(size=16)     
            ),
            plot_bgcolor="#B4C2D4", 
)

Lag_Importance_Correlations.show()

In [101]:
Lag_Importance_Correlations.write_image('PACF_Lag_Correlation_Distribution.svg')

In [ ]:
##According to the plots, Lags 1 and 7 generally are the highest correlated. 13 could be a candiate as well.

In [102]:
Lag_Importance_Correlations = px.box(
    data_frame=df_ACF_And_Pacf_Correlations,
    title='ACF Correlation Distribution Across Lags',
    x='Lag',
    y='acf',
    width=1000,
    height=900,
    color_discrete_sequence=['#1C7544']
)

conf_int = 1.96/np.sqrt(len(df_Reduced['datestamp'].unique()))

Lag_Importance_Correlations.add_hline(y=conf_int, line_dash="dot", line_color="red")
Lag_Importance_Correlations.add_hline(y=-conf_int, line_dash="dot", line_color="red")

Lag_Importance_Correlations.update_layout(
     title_font=dict(size=16),  # Set the title font size
    font=dict(size=15),  # Set general font size for labels and axis titles
            hoverlabel=dict(
                font_size=20  # Set font size for hover data
            ),
            yaxis=dict(
                title='ACF Significance Distrbution',
                title_font=dict(size=18),  # Set y-axis title font size
                tickfont=dict(size=12)  # Set y-axis tick font size
            ),
            xaxis=dict(
                title='Lags',
                title_font=dict(size=18),  # Set x-axis title font size
                tickfont=dict(size=12)  # Set x-axis tick font size
            ),
             legend=dict(
                orientation='h',       
                yanchor='bottom',     
                y=-0.3,               
                xanchor='center',      
                 x=0.5,                 
                font=dict(size=16)     
            ),
            plot_bgcolor="#B4C2D4", 
)

Lag_Importance_Correlations.show()

In [104]:
Lag_Importance_Correlations.write_image('ACF_Lag_Correlation_Distribution.svg')

In [ ]:
#According to the plot there is period importance to the lags, it goes from 1 decays then rises at 12 decays then rises at 36
#Its looks like periodic nature of 12 so we should using rolling statistics of 6, 12, 36, ann 48

Feature Generation

In [109]:
df_Reduced['datestamp'] = pd.to_datetime(df_Reduced['datestamp'])
df_Reduced = df_Reduced.sort_values(by=['unique_id','datestamp'], ascending=True)

In [ ]:
lags = [1, 7, 13]
rolling_windows = [3, 6, 12, 36, 48]
for lag in tqdm(lags):
    df_Reduced[f'y_{lag}'] = df_Reduced.groupby(['unique_id'])['y'].transform(lambda x: x.shift(lag))

    for rw in rolling_windows:

        df_Reduced[f'y_{lag}_ma_{rw}'] = df_Reduced.groupby(['unique_id'])[f'y_{lag}'].transform(lambda x: x.rolling(window=rw, min_periods=rw).mean())
        df_Reduced[f'y_{lag}_msd_{rw}'] = df_Reduced.groupby(['unique_id'])[f'y_{lag}'].transform(lambda x: x.rolling(window=rw, min_periods=rw).std())


100%|██████████| 3/3 [00:01<00:00,  1.77it/s]


In [111]:
df_Reduced

,unique_id,datestamp,y,month_in_quarter,month_in_quarter_sin,month_in_quarter_cos,month_sin,month_cos,quarter_sin,quarter_cos,...,y_13_ma_3,y_13_msd_3,y_13_ma_6,y_13_msd_6,y_13_ma_12,y_13_msd_12,y_13_ma_36,y_13_msd_36,y_13_ma_48,y_13_msd_48
4944,1,2000-01-01,12.9,1,8.660254e-01,-0.5,5.000000e-01,8.660254e-01,1.000000e+00,6.123234e-17,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4945,1,2000-02-01,14.2,2,-8.660254e-01,-0.5,8.660254e-01,5.000000e-01,1.000000e+00,6.123234e-17,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4946,1,2000-03-01,12.0,3,-2.449294e-16,1.0,1.000000e+00,6.123234e-17,1.000000e+00,6.123234e-17,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4947,1,2000-04-01,12.2,1,8.660254e-01,-0.5,8.660254e-01,-5.000000e-01,1.224647e-16,-1.000000e+00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4948,1,2000-05-01,11.8,2,-8.660254e-01,-0.5,5.000000e-01,-8.660254e-01,1.224647e-16,-1.000000e+00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69520,554,2025-05-01,18.9,2,-8.660254e-01,-0.5,5.000000e-01,-8.660254e-01,1.224647e-16,-1.000000e+00,...,26.033333,2.055075,20.966667,5.933858,20.275000,4.558533,18.841667,4.281680,18.614583,4.117942
69521,554,2025-06-01,21.8,3,-2.449294e-16,1.0,1.224647e-16,-1.000000e+00,1.224647e-16,-1.000000e+00,...,25.000000,2.628688,22.450000,4.931835,20.291667,4.569356,18.816667,4.253234,18.768750,4.144936
69522,554,2025-07-01,22.3,1,8.660254e-01,-0.5,-5.000000e-01,-8.660254e-01,-1.000000e+00,-1.836970e-16,...,21.866667,2.857155,23.116667,3.798640,20.041667,4.573333,18.752778,4.238294,18.822917,4.124924
69523,554,2025-08-01,24.5,2,-8.660254e-01,-0.5,-8.660254e-01,-5.000000e-01,-1.000000e+00,-1.836970e-16,...,20.300000,2.443358,23.166667,3.733452,19.766667,4.510862,18.730556,4.233662,18.845833,4.123877


In [115]:
df_Reduced[['unique_id','datestamp','y','y_1','y_7']].tail(10)

,unique_id,datestamp,y,y_1,y_7
69515,554,2024-12-01,17.4,15.9,23.1
69516,554,2025-01-01,17.5,17.4,18.6
69517,554,2025-02-01,14.9,17.5,19.2
69518,554,2025-03-01,16.8,14.9,15.3
69519,554,2025-04-01,16.9,16.8,16.2
69520,554,2025-05-01,18.9,16.9,15.8
69521,554,2025-06-01,21.8,18.9,15.9
69522,554,2025-07-01,22.3,21.8,17.4
69523,554,2025-08-01,24.5,22.3,17.5
69524,554,2025-09-01,25.7,24.5,14.9


Feature Selection

In [112]:
df_Reduced.columns

Index(['unique_id', 'datestamp', 'y', 'month_in_quarter',
       'month_in_quarter_sin', 'month_in_quarter_cos', 'month_sin',
       'month_cos', 'quarter_sin', 'quarter_cos', 'week', 'week_sin',
       'week_cos', 'y_1', 'y_1_ma_3', 'y_1_msd_3', 'y_1_ma_6', 'y_1_msd_6',
       'y_1_ma_12', 'y_1_msd_12', 'y_1_ma_36', 'y_1_msd_36', 'y_1_ma_48',
       'y_1_msd_48', 'y_7', 'y_7_ma_3', 'y_7_msd_3', 'y_7_ma_6', 'y_7_msd_6',
       'y_7_ma_12', 'y_7_msd_12', 'y_7_ma_36', 'y_7_msd_36', 'y_7_ma_48',
       'y_7_msd_48', 'y_13', 'y_13_ma_3', 'y_13_msd_3', 'y_13_ma_6',
       'y_13_msd_6', 'y_13_ma_12', 'y_13_msd_12', 'y_13_ma_36', 'y_13_msd_36',
       'y_13_ma_48', 'y_13_msd_48'],
      dtype='object')

In [144]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import RFE

In [145]:
lags_and_indcators = [
    'y_1', 'y_1_ma_3', 'y_1_msd_3', 'y_1_ma_6', 'y_1_msd_6',
       'y_1_ma_12', 'y_1_msd_12', 'y_1_ma_36', 'y_1_msd_36', 'y_1_ma_48',
       'y_1_msd_48', 'y_7', 'y_7_ma_3', 'y_7_msd_3', 'y_7_ma_6', 'y_7_msd_6',
       'y_7_ma_12', 'y_7_msd_12', 'y_7_ma_36', 'y_7_msd_36', 'y_7_ma_48',
       'y_7_msd_48', 'y_13', 'y_13_ma_3', 'y_13_msd_3', 'y_13_ma_6',
       'y_13_msd_6', 'y_13_ma_12', 'y_13_msd_12', 'y_13_ma_36', 'y_13_msd_36',
       'y_13_ma_48', 'y_13_msd_48'
]

exog_variables = [
    'month_in_quarter',
       'month_in_quarter_sin', 'month_in_quarter_cos', 'month_sin',
       'month_cos', 'quarter_sin', 'quarter_cos', 'week', 'week_sin',
       'week_cos'
]

categoricals = [
    'unique_id'
]

target = [
    'y'
]

In [ ]:
def Initialize_Models(n_features_to_select=10):
    """
    Initialize RandomForestRegressor with RFE wrapper
    """
    rf = RandomForestRegressor(
        n_estimators=300,      
        max_depth=8,           # limit tree depth
        min_samples_leaf=5,    # avoid overfitting
        max_features="sqrt",   # typical RF setting
        n_jobs=-1,
        random_state=42
    )

    # Recursive Feature Elimination wrapper
    rfe = RFE(
        estimator=rf,
        n_features_to_select=n_features_to_select,
        step=1
    )

    return rfe


def Run_Elimination(rfe, X, Y):
    rfe.fit(X, Y.values.ravel())
    return rfe



def Get_Sets(train, lags_and_indicators, categoricals, exog_variables, target):
    train_reduced = train[lags_and_indicators + categoricals + exog_variables + target]
    train_reduced = train_reduced.dropna(subset=lags_and_indicators + categoricals + exog_variables + target)
    X = train_reduced[lags_and_indicators + categoricals + exog_variables]
    Y = train_reduced[target]
    return X, Y


def Get_RF_Importance(rfe, X):
    rf_importance_full = np.zeros(X.shape[1])
    rf_importance_full[rfe.support_] = rfe.estimator_.feature_importances_

    summary = pd.DataFrame({
        'feature': X.columns,
        'selected': rfe.support_,
        'rfe_ranking': rfe.ranking_,
        'rf_feature_importance': rf_importance_full
    }).sort_values(
        ['selected', 'rf_feature_importance'],
        ascending=[False, False]
    )

    return summary


def Run_Wrapper_Method(train, lags_and_indicators=lags_and_indcators, categoricals=categoricals, exog_variables=exog_variables, target=target, n_features_to_select=10):

        X, Y = Get_Sets(train, lags_and_indicators, categoricals, exog_variables, target)
        rfe = Initialize_Models(n_features_to_select=n_features_to_select)
        rfe = Run_Elimination(rfe=rfe, X=X, Y=Y)
        summary = Get_RF_Importance(rfe=rfe, X=X)

        return summary



In [149]:
from dateutil.relativedelta import relativedelta

In [150]:
maxDate = df_Reduced['datestamp'].max()
offset = maxDate - relativedelta(years=4)
offset

Timestamp('2021-09-01 00:00:00')

In [151]:
train = df_Reduced[df_Reduced['datestamp'] <= offset]

In [152]:
df_Feature_Importance = Run_Wrapper_Method(train)

In [153]:
df_Feature_Importance

,feature,selected,rfe_ranking,rf_feature_importance
0,y_1,True,1,0.138911
11,y_7,True,1,0.132589
7,y_1_ma_36,True,1,0.108126
1,y_1_ma_3,True,1,0.103869
5,y_1_ma_12,True,1,0.102862
12,y_7_ma_3,True,1,0.088809
8,y_1_msd_36,True,1,0.085859
16,y_7_ma_12,True,1,0.081791
14,y_7_ma_6,True,1,0.080997
9,y_1_ma_48,True,1,0.076189


In [154]:
df_Feature_Importance.to_csv('Feature_Importance_Through_RFE.csv',index=False)